# 4-Hour Google Colab Hands-on Tutorial
## Supervised Learning Classification Project: Customer Churn Prediction

### Project Theme
You are working as a junior machine learning engineer for a telecom company.

Your task is to build a **supervised machine learning classification model** that predicts whether a customer will **churn** or **not churn** based on customer profile, usage, billing, and support-related features.

This notebook is designed for a **4-hour instructor-led hands-on session**.

## What Students Will Learn
By the end of this tutorial, learners will be able to:

1. Explain supervised learning and classification in simple terms.
2. Understand binary classification using a business example.
3. Generate and load a realistic customer churn dataset.
4. Perform exploratory data analysis for classification.
5. Handle numerical and categorical features.
6. Build preprocessing pipelines using `ColumnTransformer` and `Pipeline`.
7. Train classification models:
   - Dummy Classifier
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
8. Evaluate classification models using:
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - Confusion Matrix
   - ROC-AUC
9. Perform cross-validation and hyperparameter tuning.
10. Save the final model and predict churn for a new customer.

## 4-Hour Tutorial Agenda

| Time | Module | Focus |
|---|---|---|
| 0:00 - 0:25 | Module 1 | Introduction to supervised learning and classification |
| 0:25 - 0:55 | Module 2 | Problem statement and dataset creation |
| 0:55 - 1:35 | Module 3 | Exploratory Data Analysis for classification |
| 1:35 - 2:10 | Module 4 | Data preprocessing and train-test split |
| 2:10 - 2:55 | Module 5 | Model building: baseline, logistic regression, tree models |
| 2:55 - 3:25 | Module 6 | Model evaluation and comparison |
| 3:25 - 3:50 | Module 7 | Hyperparameter tuning and model saving |
| 3:50 - 4:00 | Module 8 | Final discussion, assignment, and Q&A |

# Module 1: Supervised Learning and Classification

## What is supervised learning?
Supervised learning means we train a machine learning model using examples where the correct answer is already known.

Example:

| Input Data | Correct Answer |
|---|---|
| Customer tenure, monthly charges, support calls | Churn or Not Churn |
| Email text | Spam or Not Spam |
| Patient test values | Disease or No Disease |

The model learns the relationship between input features and the target output.

## What is classification?
Classification is a type of supervised learning where the target value is a category/class.

Examples:

- Churn / Not Churn
- Fraud / Not Fraud
- Spam / Not Spam
- Approved / Rejected
- Disease / No Disease

In this project, we will predict whether a customer will churn, so this is a **binary classification problem**.

## Business Problem
A telecom company is losing customers every month. Retaining an existing customer is usually cheaper than acquiring a new customer.

The business wants to identify customers who are likely to leave, so the retention team can offer discounts, support, or personalized plans.

## ML Problem
Build a supervised classification model to predict:

```text
Churn = 1 → Customer is likely to leave
Churn = 0 → Customer is likely to stay
```

## Success Criteria
A good model should:

- Identify churn customers correctly.
- Reduce missed churn cases.
- Provide reliable probability scores.
- Perform better than a simple baseline model.
- Generalize well on unseen customer data.

# Module 2: Environment Setup
Run the following cell in Google Colab.

In [ ]:
# Install required libraries
!pip -q install pandas numpy matplotlib scikit-learn joblib

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings

warnings.filterwarnings("ignore")

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

print("Libraries imported successfully")

# Create a Realistic Customer Churn Dataset

To make this notebook easy to run in Google Colab, we will generate a realistic synthetic telecom churn dataset.

This dataset will include:

| Column | Meaning |
|---|---|
| Customer_ID | Unique customer ID |
| Age | Customer age |
| Tenure_Months | Number of months with company |
| Monthly_Charges | Monthly bill amount |
| Total_Charges | Total amount paid so far |
| Contract_Type | Month-to-month, One year, Two year |
| Internet_Service | Fiber optic, DSL, No |
| Payment_Method | Payment method |
| Support_Calls | Number of support calls |
| Data_Usage_GB | Monthly data usage |
| Late_Payments | Number of late payments |
| Churn | Target variable: 1 = Churn, 0 = Not Churn |

In [ ]:
# Generate a synthetic telecom churn dataset
np.random.seed(42)

n = 5000

age = np.random.randint(18, 75, n)
tenure_months = np.random.randint(1, 73, n)
monthly_charges = np.round(np.random.normal(70, 25, n), 2)
monthly_charges = np.clip(monthly_charges, 20, 150)

data_usage_gb = np.round(np.random.gamma(shape=4, scale=12, size=n), 2)
support_calls = np.random.poisson(lam=1.5, size=n)
late_payments = np.random.poisson(lam=0.7, size=n)

contract_type = np.random.choice(
    ["Month-to-month", "One year", "Two year"],
    size=n,
    p=[0.55, 0.25, 0.20]
)

internet_service = np.random.choice(
    ["Fiber optic", "DSL", "No"],
    size=n,
    p=[0.50, 0.35, 0.15]
)

payment_method = np.random.choice(
    ["Electronic check", "Credit card", "Bank transfer", "Mailed check"],
    size=n,
    p=[0.40, 0.25, 0.25, 0.10]
)

total_charges = np.round(monthly_charges * tenure_months + np.random.normal(0, 100, n), 2)
total_charges = np.clip(total_charges, 0, None)

# Create churn probability using business-like rules
base_logit = (
    -2.0
    + 0.030 * monthly_charges
    - 0.035 * tenure_months
    + 0.280 * support_calls
    + 0.350 * late_payments
    + 0.005 * age
    + np.where(contract_type == "Month-to-month", 1.0, 0)
    + np.where(contract_type == "Two year", -0.9, 0)
    + np.where(internet_service == "Fiber optic", 0.45, 0)
    + np.where(payment_method == "Electronic check", 0.55, 0)
)

prob_churn = 1 / (1 + np.exp(-base_logit))
churn = np.random.binomial(1, prob_churn)

customer_df = pd.DataFrame({
    "Customer_ID": [f"CUST_{i:05d}" for i in range(1, n + 1)],
    "Age": age,
    "Tenure_Months": tenure_months,
    "Monthly_Charges": monthly_charges,
    "Total_Charges": total_charges,
    "Contract_Type": contract_type,
    "Internet_Service": internet_service,
    "Payment_Method": payment_method,
    "Support_Calls": support_calls,
    "Data_Usage_GB": data_usage_gb,
    "Late_Payments": late_payments,
    "Churn": churn
})

customer_df.head()

In [ ]:
# Check shape of dataset
print("Number of rows:", customer_df.shape[0])
print("Number of columns:", customer_df.shape[1])

## Student Task 1
Answer these questions:

1. What is the target variable?
2. Is this a classification or regression problem?
3. Why is customer churn prediction a classification problem?
4. Which columns are numerical?
5. Which columns are categorical?

# Module 3: Exploratory Data Analysis

Exploratory Data Analysis, or EDA, means understanding the data before building the model.

For classification problems, we usually check:

- Target class distribution
- Missing values
- Numeric feature distributions
- Categorical feature patterns
- Relationship between features and target
- Class imbalance

In [ ]:
# Basic information
customer_df.info()

In [ ]:
# Check missing values
customer_df.isnull().sum()

In [ ]:
# Summary statistics for numeric columns
customer_df.describe().T

## Target Variable Distribution

Before model building, check how many customers churn and how many do not churn.

If one class is much larger than the other, the dataset is imbalanced.

In [ ]:
# Count churn vs non-churn customers
churn_counts = customer_df["Churn"].value_counts().sort_index()
churn_counts

In [ ]:
# Plot target distribution
plt.figure(figsize=(6, 4))
plt.bar(["Not Churn", "Churn"], churn_counts.values)
plt.xlabel("Customer Class")
plt.ylabel("Number of Customers")
plt.title("Target Distribution: Churn vs Not Churn")
plt.show()

print("Churn percentage:", round(customer_df["Churn"].mean() * 100, 2), "%")

## Feature Distribution
Let us visualize a few important numeric features.

In [ ]:
selected_numeric_features = ["Age", "Tenure_Months", "Monthly_Charges", "Support_Calls", "Late_Payments"]

for col in selected_numeric_features:
    plt.figure(figsize=(7, 4))
    plt.hist(customer_df[col], bins=30)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    plt.show()

## Relationship Between Features and Churn
We will compare average churn rate across different groups.

In [ ]:
# Churn rate by contract type
contract_churn = customer_df.groupby("Contract_Type")["Churn"].mean().sort_values(ascending=False)
contract_churn

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(contract_churn.index, contract_churn.values)
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate")
plt.title("Churn Rate by Contract Type")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Churn rate by payment method
payment_churn = customer_df.groupby("Payment_Method")["Churn"].mean().sort_values(ascending=False)
payment_churn

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(payment_churn.index, payment_churn.values)
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate")
plt.title("Churn Rate by Payment Method")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Average numeric values by churn class
customer_df.groupby("Churn")[["Age", "Tenure_Months", "Monthly_Charges", "Support_Calls", "Late_Payments"]].mean()

## Student Task 2
Based on EDA, answer:

1. Which contract type has the highest churn rate?
2. Which payment method has the highest churn rate?
3. Do customers with more support calls churn more?
4. Do customers with shorter tenure churn more?
5. Is the dataset heavily imbalanced?

# Module 4: Feature Engineering and Preprocessing

Most machine learning models cannot directly understand text categories such as:

```text
Month-to-month
Fiber optic
Electronic check
```

So we need preprocessing.

| Column Type | Technique |
|---|---|
| Numeric columns | StandardScaler |
| Categorical columns | OneHotEncoder |
| ID column | Drop |
| Target column | Keep separately |

In [ ]:
# Define target and features
target = "Churn"

X = customer_df.drop(columns=[target, "Customer_ID"])
y = customer_df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## Train-Test Split

We split data into:

- **Training data:** used to train the model
- **Testing data:** used to check model performance on unseen data

We use `stratify=y` to maintain similar churn percentage in both training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training churn rate:", round(y_train.mean() * 100, 2), "%")
print("Testing churn rate:", round(y_test.mean() * 100, 2), "%")

In [ ]:
# Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created")

## Student Task 3
Answer:

1. Why did we drop `Customer_ID`?
2. Why do we use `OneHotEncoder` for categorical columns?
3. Why do we use `StandardScaler` for numeric columns?
4. Why did we use `stratify=y` in train-test split?
5. Why is a pipeline useful in real ML projects?

# Module 5: Model Building

We will train multiple classification models:

1. Dummy Classifier baseline
2. Logistic Regression
3. Decision Tree Classifier
4. Random Forest Classifier

The goal is to compare models and select the best one.

## Evaluation Function

We will create a function to evaluate every model consistently.

Classification metrics:

| Metric | Meaning |
|---|---|
| Accuracy | Overall correct predictions |
| Precision | Out of predicted churn customers, how many actually churned |
| Recall | Out of actual churn customers, how many we caught |
| F1-score | Balance between precision and recall |
| ROC-AUC | Ability to separate churn and not churn customers |

For churn prediction, **recall is very important** because missing a churn customer can be costly for business.

In [ ]:
def evaluate_classification_model(model, X_test, y_test, model_name):
    """Evaluate a classification model and return key metrics."""
    y_pred = model.predict(X_test)

    # Some models provide probability scores
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc
    }

    return results

## 1. Baseline Model: Dummy Classifier
A baseline model gives a simple reference point.

Here, the dummy classifier predicts the most frequent class.

Any useful ML model should perform better than this baseline.

In [ ]:
dummy_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_model.fit(X_train, y_train)

dummy_results = evaluate_classification_model(
    dummy_model, X_test, y_test, "Dummy Classifier"
)

dummy_results

## 2. Logistic Regression
Logistic Regression is a simple and popular classification model.

Despite the name regression, it is commonly used for classification problems.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_model.fit(X_train, y_train)

logistic_results = evaluate_classification_model(
    logistic_model, X_test, y_test, "Logistic Regression"
)

logistic_results

## 3. Decision Tree Classifier
A Decision Tree asks question-based rules, like a flowchart.

Example:

```text
Is contract month-to-month?
Is monthly charge high?
Did the customer make support calls?
```

Then it predicts churn or not churn.

In [ ]:
decision_tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=6, random_state=42))
])

decision_tree_model.fit(X_train, y_train)

decision_tree_results = evaluate_classification_model(
    decision_tree_model, X_test, y_test, "Decision Tree"
)

decision_tree_results

## 4. Random Forest Classifier
Random Forest creates many decision trees and combines their predictions.

Layman meaning:

```text
Decision Tree = one expert
Random Forest = group of experts
```

It is usually more stable than a single decision tree.

In [ ]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

random_forest_results = evaluate_classification_model(
    random_forest_model, X_test, y_test, "Random Forest"
)

random_forest_results

# Module 6: Model Evaluation and Comparison

Now we compare all models in one table.

In [ ]:
results_df = pd.DataFrame([
    dummy_results,
    logistic_results,
    decision_tree_results,
    random_forest_results
])

results_df.sort_values(by="F1 Score", ascending=False)

## Confusion Matrix

A confusion matrix shows where the model is correct and where it is wrong.

For churn prediction:

| Term | Meaning |
|---|---|
| True Negative | Correctly predicted not churn |
| True Positive | Correctly predicted churn |
| False Positive | Predicted churn but customer did not churn |
| False Negative | Predicted not churn but customer actually churned |

For churn business problems, **False Negative is risky** because the company misses a customer who is likely to leave.

In [ ]:
# Confusion matrix for Random Forest
rf_pred = random_forest_model.predict(X_test)

cm = confusion_matrix(y_test, rf_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Churn", "Churn"])
disp.plot(values_format="d")
plt.title("Confusion Matrix - Random Forest")
plt.show()

In [ ]:
# Classification report
print(classification_report(y_test, rf_pred, target_names=["Not Churn", "Churn"]))

## ROC Curve
ROC curve helps us understand how well the model separates churn from not churn.

Higher ROC-AUC means better separation ability.

In [ ]:
RocCurveDisplay.from_estimator(random_forest_model, X_test, y_test)
plt.title("ROC Curve - Random Forest")
plt.show()

## Student Task 4
Answer:

1. Which model has the highest accuracy?
2. Which model has the highest recall?
3. Which model has the highest F1-score?
4. For churn prediction, why can recall be more important than accuracy?
5. Which model would you choose and why?

# Cross Validation

Normal train-test split tests the model once.

Cross-validation tests the model multiple times on different parts of the data.

This gives a more reliable performance estimate.

In [ ]:
# Cross-validation using F1 score
cv_scores = cross_val_score(
    random_forest_model,
    X,
    y,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

print("Cross-validation F1 scores:", cv_scores)
print("Average CV F1 score:", round(cv_scores.mean(), 4))
print("Standard deviation:", round(cv_scores.std(), 4))

# Module 7: Hyperparameter Tuning

Hyperparameter tuning means finding the best settings for a model.

For Random Forest, we can tune:

| Hyperparameter | Meaning |
|---|---|
| n_estimators | Number of trees |
| max_depth | Maximum depth of each tree |
| min_samples_split | Minimum samples needed to split a node |
| min_samples_leaf | Minimum samples needed in a final leaf |

We will use `GridSearchCV`.

In [ ]:
# Smaller grid for 4-hour class runtime
param_grid = {
    "model__n_estimators": [100, 150],
    "model__max_depth": [5, 8, 12],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    estimator=random_forest_model,
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV F1 score:", round(grid_search.best_score_, 4))

In [ ]:
# Evaluate tuned model
best_model = grid_search.best_estimator_

tuned_results = evaluate_classification_model(
    best_model,
    X_test,
    y_test,
    "Tuned Random Forest"
)

tuned_results

In [ ]:
# Add tuned model to comparison table
final_results_df = pd.concat([
    results_df,
    pd.DataFrame([tuned_results])
], ignore_index=True)

final_results_df.sort_values(by="F1 Score", ascending=False)

# Feature Importance

Feature importance tells us which features were most useful for prediction.

This works well with tree-based models like Random Forest.

In [ ]:
# Extract feature names after preprocessing
preprocessor_fitted = best_model.named_steps["preprocessor"]
model_fitted = best_model.named_steps["model"]

# Numeric feature names
num_features = numeric_features

# Categorical feature names after one-hot encoding
cat_encoder = preprocessor_fitted.named_transformers_["cat"].named_steps["onehot"]
cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()

all_feature_names = num_features + cat_features

# Feature importance table
importance_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": model_fitted.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance_df.head(10)

In [ ]:
# Plot top 10 important features
top_features = importance_df.head(10).sort_values(by="Importance")

plt.figure(figsize=(8, 5))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importances")
plt.show()

# Save the Final Model

After selecting the best model, save it using `joblib`.

This saved model can later be used in a Streamlit app, Flask API, FastAPI app, or production system.

In [ ]:
# Save final model
model_filename = "customer_churn_classification_model.pkl"
joblib.dump(best_model, model_filename)

print("Model saved as:", model_filename)

In [ ]:
# Optional: download model file in Google Colab
try:
    from google.colab import files
    files.download(model_filename)
except Exception as e:
    print("Download works only in Google Colab.")

# Make Prediction for a New Customer

Now let us predict whether a new customer will churn.

In [ ]:
new_customer = pd.DataFrame({
    "Age": [35],
    "Tenure_Months": [6],
    "Monthly_Charges": [110.0],
    "Total_Charges": [660.0],
    "Contract_Type": ["Month-to-month"],
    "Internet_Service": ["Fiber optic"],
    "Payment_Method": ["Electronic check"],
    "Support_Calls": [4],
    "Data_Usage_GB": [75.0],
    "Late_Payments": [2]
})

prediction = best_model.predict(new_customer)[0]
probability = best_model.predict_proba(new_customer)[0][1]

print("Prediction:", "Churn" if prediction == 1 else "Not Churn")
print("Churn Probability:", round(probability * 100, 2), "%")

## Business Interpretation

If the model predicts a high churn probability, the retention team can take action:

- Offer discount
- Call the customer
- Resolve service issue
- Offer better plan
- Provide loyalty reward

Machine learning helps the business act before the customer leaves.

# Module 8: Final Assignment

## Assignment Problem
Improve the churn prediction model by trying the following:

1. Add a new feature called `High_Value_Customer`.
   - If monthly charges are greater than 90, mark as 1, else 0.
2. Train Logistic Regression and Random Forest again.
3. Compare old and new model performance.
4. Tune Random Forest with a larger parameter grid.
5. Try changing the classification threshold from 0.5 to 0.4.
6. Observe whether recall improves.
7. Write a short business recommendation.

## Submission Requirements
Students should submit:

1. Completed Colab notebook
2. Model comparison table
3. Confusion matrix
4. ROC curve
5. Feature importance chart
6. Final business interpretation

# Viva / Interview Questions

1. What is supervised learning?
2. What is classification?
3. Why is churn prediction a classification problem?
4. What is the difference between binary and multiclass classification?
5. What is accuracy?
6. What is precision?
7. What is recall?
8. What is F1-score?
9. What is confusion matrix?
10. Why can accuracy be misleading in imbalanced datasets?
11. What is ROC-AUC?
12. What is cross-validation?
13. What is hyperparameter tuning?
14. Why is Random Forest usually more stable than Decision Tree?
15. How can this model help a telecom business?

# Final Summary

In this 4-hour hands-on project, we built an end-to-end supervised learning classification solution.

We followed this full ML lifecycle:

```text
Problem understanding
Dataset creation
EDA
Preprocessing
Train-test split
Model building
Model evaluation
Cross-validation
Hyperparameter tuning
Feature importance
Model saving
New prediction
Business interpretation
```

This is the same flow used in real-world classification projects.